# Wisdom Tooth Resorption Detection — Results

Standalone view of [`results/metrics.json`](../results/metrics.json) and the
per-model dumps under [`results/metrics/`](../results/metrics/). Renders the
baseline-vs-improved comparison, per-class F1 bars and the confusion matrix
produced by the latest evaluation run.

Run `python -m tooth_resorption.training.train --data synthetic` followed by
`python -m tooth_resorption.evaluation.evaluate --data synthetic` first if you
want `results/plots/confusion_matrix.png` to reflect a freshly trained model.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
metrics = json.loads((ROOT / 'results' / 'metrics.json').read_text(encoding='utf-8'))
metrics['_meta']

## Head-to-head metrics

In [ ]:
rows = []
for variant in ('baseline', 'improved'):
    row = {k: v for k, v in metrics[variant].items() if isinstance(v, (int, float))}
    row['variant'] = variant
    row['model'] = metrics[variant]['model']
    rows.append(row)
df = pd.DataFrame(rows).set_index('variant')
df[['model', 'f1_macro', 'precision_macro', 'recall_macro', 'accuracy', 'auc_macro', 'inference_time_ms']]

In [ ]:
from tooth_resorption.visualization.plot_comparison import render
from IPython.display import Image

render()
Image(filename=str(ROOT / 'results' / 'plots' / 'comparison.png'))

## Per-class F1 (improved model)

In [ ]:
per_class = metrics['improved']['per_class_f1']
fig, ax = plt.subplots(figsize=(6.0, 3.6))
labels = list(per_class.keys())
values = list(per_class.values())
bars = ax.bar(labels, values, color=['#3b7dbf', '#5fa1d9', '#7fbfe5'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 score')
ax.set_title('Per-class F1 — ViT-Base/16 on the real test set')
ax.grid(axis='y', linestyle=':', alpha=0.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## Inference latency

In [ ]:
times = [metrics['baseline']['inference_time_ms'], metrics['improved']['inference_time_ms']]
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.bar(['BaselineCNN', 'ViT-Base/16'], times, color=['#bd6760', '#3b7dbf'])
ax.set_ylabel('ms / image')
ax.set_title('Inference latency (single-image, batch=1)')
for i, v in enumerate(times):
    ax.text(i, v + 1, f'{v:.1f} ms', ha='center')
ax.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Confusion matrix (most recent evaluation run)

In [ ]:
cm_path = ROOT / 'results' / 'plots' / 'confusion_matrix.png'
if cm_path.exists():
    from IPython.display import Image as _Image
    display(_Image(filename=str(cm_path)))
else:
    print('Run `python -m tooth_resorption.evaluation.evaluate --data synthetic` to produce the confusion matrix.')

## Notes

- The numbers above are from the original MSc experiments on the private clinical dataset (Mersin University) — they are NOT reproduced by the synthetic smoke run.
- The published baseline collapsed to predicting only the majority class — confirming that the ~55-image dataset is too small to train a CNN from scratch.
- The improved model is a Vision Transformer fine-tuned from ImageNet-21k weights; it was the best of 10 attention/transformer variants evaluated.